# Hierarchical Document Refinement for Long-context RAG

In [12]:
from IPython.core.display import HTML
HTML("<style>.rendered_html, .cm-content { font-family: 'Times New Roman'; font-size: 22px !important; line-height: 30px;  }</style>")
# %%html
# <style>
# .vbox, .MathJax_Display, .MathJax_CHTML, .MathJax_SVG_Display {
#     text-align: left !important;
#     justify-content: flex-start !important;
# }
# </style>

In [ ]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel

## Problem Description

Real-world RAG applications often encounter long-context input scenarios, where redundant and noisy information results in higher inference costs and reduced performance. Although these documents contain the necessary information for generating accurate responses, their extensive length poses two primary challenges for practical RAG deployments:
<ol><li><b>Signal-to-noise ratio:</b> Long documents often contain substantial irrelevant content alongside pertinent information, making it difficult for models to focus on query-relevant details</li>
    <li><b>Computational overhead:</b> Processing complete documents significantly increases the input context length, resulting in higher computational costs and potential performance bottlenecks in production environments</li>
</ol>

## Solution

<p>To address these 2 challenges, an intuitive approach is to refine the retrieved long form documents before LLM processing.</p>

<p>A universal document-level refinement framework called LongRefiner, achieves efficient, low-latency long-text refinement by 
leveraging hierarchical textual information. It works on the principle that complete documents retain the rich structural 
information like logical connections and content organization, which enables more precise information extraction than 
traditional chunk-based approaches.</p>

<p>LongRefiner performs efficient document refinement by modeling structural information in long documents.</p>

## LongRefiner: a Long Document Refiner for RAG

This is a 3 step method. Each step has been described below along with its code snippet

Following is the Prompt formation utility class used in all the 3 Steps.

#### Prompt handler

In [ ]:
class PromptTemplate:
    placeholders = ["reference", "question"]
    base_system_prompt = (
        "Answer the question based on the given document."
        "Only give me the answer and do not output any other words."
        "\nThe following are given documents.\n\n{reference}"
    )
    base_user_prompt = "Question: {question}"

    def __init__(self, tokenizer, system_prompt="", user_prompt=""):

        self.max_input_len = 64000
        self.tokenizer = tokenizer

        if len(system_prompt) == 0 and len(user_prompt) == 0:
            system_prompt = self.base_system_prompt
            user_prompt = self.base_user_prompt
        self.system_prompt = system_prompt
        self.user_prompt = user_prompt
        self.enable_chat = True
        self.is_chat = True
        self.is_openai = False
        self.reference_template = None

        # self._check_placeholder()

    def _check_placeholder(self):
        # check placeholder in prompt
        for holder in self.placeholders:
            flag = False
            for prompt in [self.system_prompt, self.user_prompt]:
                if f"{holder}" in prompt:
                    print(f"Find `{holder}` in template")
                    flag = True
                    break
            if not flag and holder != "reference":
                assert False

    def truncate_prompt(self, prompt):
        if self.is_openai:
            truncated_messages = []
            total_tokens = 0
            assert isinstance(prompt, list)
            for message in prompt:
                role_content = message["content"]
                encoded_message = self.tokenizer.encode(role_content)

                if total_tokens + len(encoded_message) <= self.max_input_len:
                    truncated_messages.append(message)
                    total_tokens += len(encoded_message)
                else:
                    print(
                        f"""The input text length is greater than the maximum 
                        length ({total_tokens + len(encoded_message)} > {self.max_input_len}) and 
                        has been truncated!"""
                    )
                    remaining_tokens = self.max_input_len - total_tokens
                    truncated_message = self.encoding.decode(encoded_message[:remaining_tokens])
                    message["content"] = truncated_message
                    truncated_messages.append(message)
                    break

            return truncated_messages

        else:
            assert isinstance(prompt, str)
            tokenized_prompt = self.tokenizer(prompt, truncation=False, return_tensors="pt").input_ids[0]

            if len(tokenized_prompt) > self.max_input_len:
                print(
                    f"""The input text length is greater 
                    than the maximum length ({len(tokenized_prompt)} > {self.max_input_len}) and 
                    has been truncated!"""
                )
                half = int(self.max_input_len / 2)
                prompt = self.tokenizer.decode(
                    tokenized_prompt[:half], skip_special_tokens=True
                ) + self.tokenizer.decode(tokenized_prompt[-half:], skip_special_tokens=True)
            return prompt

    def get_prompt(self, question=None, retrieval_result=None, formatted_reference=None, 
                   previous_gen=None, messages=None, **params):
        if messages is not None:
            if isinstance(messages, str):
                return self.truncate_prompt(messages)
            if self.is_chat and self.enable_chat:
                if self.is_openai:
                    self.truncate_prompt(messages)
                else:
                    prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                    return self.truncate_prompt(prompt)
            else:
                prompt = "\n\n".join([message["content"] for message in messages if message["content"]])
                return self.truncate_prompt(prompt)

        if formatted_reference is None:
            if retrieval_result is not None:
                formatted_reference = self.format_reference(retrieval_result)
            else:
                formatted_reference = ""

        input_params = {"question": question, "reference": formatted_reference}
        input_params.update(**params)

        system_prompt = self.system_prompt.format(**input_params)
        user_prompt = self.user_prompt.format(**input_params)

        if self.is_chat and self.enable_chat:
            input = []
            if system_prompt != "":
                input.append({"role": "system", "content": system_prompt})
            if user_prompt != "":
                input.append({"role": "user", "content": user_prompt})
            if not self.is_openai:
                input = self.tokenizer.apply_chat_template(input, tokenize=False, add_generation_prompt=True)
        else:
            input = "\n\n".join([prompt for prompt in [system_prompt, user_prompt] if prompt != ""])

        if previous_gen is not None and previous_gen not in ["", " "] and self.is_openai is False:
            input += previous_gen

        return self.truncate_prompt(input)

    def format_reference(self, retrieval_result):
        format_reference = ""
        for idx, doc_item in enumerate(retrieval_result):
            content = doc_item["contents"]
            title = content.split("\n")[0]
            text = "\n".join(content.split("\n")[1:])
            if self.reference_template is not None:
                format_reference += self.reference_template.format(idx=idx, title=title, text=text)
            else:
                format_reference += f"Doc {idx+1}(Title: {title}) {text}\n"

        return format_reference

### Dual-Level Query Analysis

Answering Queries requires diverse information collection from simple facts to complex reasoning. This diversity is characterized by 2 Information Levels
<ul><li><b>Local Level:</b> This refers to knowledge confined to specific contexts or localized information, involving a narrow knowledge scope such as a single passage or snippet.</li>
    <li><b>Global Level:</b> This encompasses broad, comprehensive knowledge, involving a wide range of information and
background context.</li></ul>

These levels are <b>quantified</b>, using a dual-level query analyzer. A teacher LLM (Llama3.1-70B-Instruct) is first prompted with task-specific instructions to analyze each query in the training dataset and assigned a corresponding information level, which is represented as a binary label.

In dual-level query analysis the training labels are annotated using . The prompt used for annotation is shown below.

In [9]:
SYSTEM_PROMPT_STEP1 = """\nYou are an assistant that performs step-by-step analysis of user queries.

**Instructions for Query Analysis:**
When given a query, please **understand the query intents**, and classify the query as either [Local] or [Global].

- [Global]: The query requires a broad or vague range of knowledge (e.g., summary or open-ended questions), 
  and may require a comprehensive understanding of the document.
- [Local]: The query has a clear and fixed answer with a narrow scope of knowledge (e.g., factual questions), 
  and only a small amount of text fragments are needed to answer.

**Output Format:**
Please present the results in JSON format with the following keys:
**query_type**: [Local] or [Global]

**Demonstration**
{demonstrations}

Query: {query}
Results:
"""

USER_PROMPT_STEP1 = """\n{question}"""

SYSTEM_PROMPT_STEP2 = "Divide the following long text into well-structured, appropriately sized chapters."
USER_PROMPT_STEP2 = "{doc_content}"

SYSTEM_PROMPT_STEP3 = """You will be provided with three inputs:
1. An user question that may need a long-form detail answer.
2. The abstract of a document
3. Outline of the document, contains titles of section and subsections in the document.

Your task is to understand the article based on its abstract and outline, and select all the parts that are helpful for answering questions (provide corresponding titles, or `abstract`).
"""

USER_PROMPT_STEP3 = (
    "**Document abstract**: {abstract}\n**Document outline**: {outline}\n**Question**:{question}\nOutput:"
)

In [ ]:
base_model_path: str = "Qwen/Qwen2.5-3B-Instruct",
query_analysis_module_lora_path: str = "",
doc_structuring_module_lora_path: str = "",
global_selection_module_lora_path: str = "",
score_model_name: str = "bge-reranker-v2-m3",
score_model_path: str = "BAAI/bge-reranker-v2-m3",
max_model_len: int = 25000

In [ ]:
model = LLM(base_model_path, enable_lora=True, max_model_len=max_model_len, gpu_memory_utilization=0.7)
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

The labels are treated as Special token and the refiner is finetuned to generate the corresponding special token based on the input query. During inference, LongRefiner adaptively determines the amount of information required for each query by predicting the appropriate information-level token. 

The formula is written as:


$$
\begin{aligned}
P_\mathcal{l} &= P_\mathcal{M}(\text{Local | query}) \\
P_\mathcal{g} &= P_\mathcal{M}(\text{Global | query}) \\
R_q &= \text{Softmax}(P_l, P_g)_g
\end{aligned}
$$

Given an array of log probabilities $L$, the Softmax probability $P_{i}$ for the $i$-th class is calculated by taking the exponential of that log probability: $P_{i}=e^{L_{i}}$. This is the approach taken in the following code snippet for Query Analysis.

In [ ]:
def run_query_analysis(question_list: List[str]) -> List[dict]:
    """
    Args:
        question_list: List[str], each question is a string
    Output:
        List[dict], each dict contains the local and global information of the question
    """
    # get special token id
    special_token = ["Local", "Global"]
    id2special = {tokenizer.encode(token)[0]: token for token in special_token}
    # get prompt template, sampling params, lora request
    prompt_template = PromptTemplate(tokenizer, system_prompt=SYSTEM_PROMPT_STEP1, user_prompt=USER_PROMPT_STEP1)
    sampling_params = SamplingParams(temperature=0, max_tokens=2, logprobs=20)
    lora_request = LoRARequest(lora_name="query_analysis", lora_int_id=1, lora_path=query_analysis_module_lora_path)

    prompt_list = [get_prompt(question=question) for question in question_list]
    output_list = model.generate(prompt_list, sampling_params=sampling_params, lora_request=lora_request)

    query_analysis_result = []
    for output in output_list:
        # set init prob for special token
        special_token_prob = {token: -100 for token in special_token}
        logprobs = output.outputs[0].logprobs[1]
        for token_id, logprob in logprobs.items():
            if token_id in id2special:
                special_token_prob[id2special[token_id]] = logprob.logprob
        for k, v in special_token_prob.items():
            special_token_prob[k] = np.exp(v)
        query_analysis_result.append(special_token_prob)
    return query_analysis_result

To achieve a more precise representation of the information scope rather than a simple binary label, Softmax is applied to the generation probabilities of the two-level tokens, producing a continuous representation $R_q$ as the information scope for the query.

### Hierarchical Document Structuring

To facilitate efficient refinement, the unstructured retrieved documents are converted into a hierarchical format with a clear article outline, hierarchical organization, and paragraph segmentation. The hierarchical structure is defined as follows.

<b>Tree-based structured document definition.</b> The document structure is modeled as a doc tree. Formally, the structured representation of a document $\mathcal{D}$ is denoted as $\mathcal{D}_{str} = (\mathcal{N},\mathcal{R})$, where $\mathcal{N}$ is the set of nodes in the document like a section, subsection, or paragraph, with its corresponding title and content. R denotes the set of relations, which capture the implication and hierarchical relationships between nodes.

To derive such a structured representation from the original long document, we leverage a long-context window LLM as our backbone. However, this task still poses two significant challenges: 
<ol><li>As $\mathcal{D}_{str}$ introduces additional information such as titles and structural information, the number of tokens required to represent it is greater than the tokens in $\mathcal{D}$ itself, which results in an excessively
    long learning target.</li> 
<li>The retrieved documents consist only of plain text without the full structure or outline, lacking the golden labels needed for training.</li></ol>

To address these challenges, 2 novel designs are proposed.

<b>XML-format document structure representation.</b> An XML-format doc structuring method is used to address the problem of a document being too long and difficult to learn. First a flat representation $D_{xml}$ of $D_{str}$, which corresponds to $D_{str}$ one-to-one, enabling the complete representation of the document’s overall structure with fewer tokens.

$D_{xml}$ in an XML-like format with special tags to represent the hierarchical structure of the document. These 3 types of tags to denote the document structure are: $\textbf{<section: {title}>}$ , $\textbf{<subsection: {title}>}$ , and $\textbf{<br>}$. Each section's content is enclosed within $\textbf{<section: {title}>}$ and $\textbf{</section: {title}>}$ , with a corresponding title that conveys the general meaning of the section. Within each section, there may be several subsections, each enclosed by  $\textbf{<subsection: {title}>}$ and  $\textbf{</subsection: {title}>}$, with a corresponding subsection title. Additionally, each section and subsection may contain its content, which is also enclosed within the tags mentioned above. Within the content, $\textbf{<br>}$ is used to denote paragraph segmentation. With this representation, a structured document tree is converted into a flat textual form. Furthermore, using parsing algorithms, this flat representation can be converted back into the original document tree without losing information.

Since most tokens in the flat representation are derived from the original document, redundant content can be omitted and restored during structure recovery. A $\textbf{<skip>}$ tag is used to represent omitted content, retaining only the first and last $k$ tokens of each paragraph while replacing the omitted content with $\textbf{<skip>}$. This results in the final
$D_{xml}$. The parameter $k$ is a hyperparameter, where a smaller k reduces the token count but increases parsing errors. The XML-based $D_{xml}$ reduces the token count to approximately $\frac{1}{10}$ of the original while preserving the original information. Then a long-context LLM is trained to learn the mapping from the original document $D$ to $D_{xml}$. Based on this learning objective, the model needs to learn how to segment, organize and summarize the total document. Then in the inference stage, we can map predicted $D_{xml}$ to $D_{str}$ by using the original document and parsing algorithm, obtaining the document tree. The overall inference process can be described as follows:

<b>(1) Structure generation.</b> The model first generates the hierarchical structure $S$ (e.g., $\textbf{<section:{title}>}$ with a suitable title):

$$
\begin{aligned}
P_\mathcal{M}(\textbf{<section: {title}>} | D) = \prod_{t=1}^{T_s} P_\mathcal{M}(y_t | y_{1:t-1}, D)
\end{aligned}
$$
    
where the title is automatically generated and fully predicted by the model based on the document.    


<b>(2) Content filling based on structure</b> Then the model generates the content $C_i$ of each part based on the corresponding $\textbf{</section: {title}>}$ and original document. The content generation probability can be expressed as:


$$
\begin{aligned}
P_\mathcal{M}(C| \textbf{<section: {title}>} | D) = P_\mathcal{M}(C^{:k}, \textbf{<skip>}, C^{k:})
\end{aligned}
$$

The model will dynamically skip the middle portion, preserving only the first $k$ and last $k$ tokens, and marking the skipped portion with $\textbf{<skip>}$.

In [ ]:
def run_doc_structuring(self, document_list: List[List[dict]]) -> List[List[dict]]:
    """
    Args:
        document_list: List[List[dict]], each item is a list of documents, each document is a dictionary, 
        contains a key 'content': "{title}\n{content}"
    Output:
        List[List[dict]], each item is a list of documents, each document is a dictionary, 
        contains the structured content of the document
    """
    # get prompt template, sampling params, lora request
    prompt_template = PromptTemplate(tokenizer, system_prompt=SYSTEM_PROMPT_STEP2, user_prompt=USER_PROMPT_STEP2)
    sampling_params = SamplingParams(temperature=0, max_tokens=10000)
    lora_request = LoRARequest(lora_name="doc_structuring", lora_int_id=2, lora_path=doc_structuring_module_lora_path)

    # get doc content (doc title is not used)
    doc_content_list = [
        ["\n".join(doc["contents"].split("\n")[1:]) for doc in item_doc_list] for item_doc_list in document_list
    ]
    # truncate doc content if it is too long
    doc_content_list = [
        [truncate_doc_content(doc_content) for doc_content in item_doc_content_list]
        for item_doc_content_list in doc_content_list
    ]

    # for each doc, get the input prompt and output the structured content
    prompt_list = sum(
        [
            [prompt_template.get_prompt(doc_content=doc_content) for doc_content in zip(item_doc_content_list)]
            for item_doc_content_list in doc_content_list
        ],
        [],
    )
    output_list = model.generate(prompt_list, sampling_params=sampling_params, lora_request=lora_request)

    # parse output to structured content
    structured_doc_list = []
    start_idx = 0
    for idx, item_doc_list in enumerate(doc_content_list):
        item_structured_docs = []
        for doc_idx, doc_content in enumerate(item_doc_list):
            output = output_list[start_idx]
            doc_title = document_list[idx][doc_idx]["contents"].split("\n")[0]
            structured_doc = parse_xml_doc(doc_content, output.outputs[0].text)
            structured_doc["title"] = doc_title
            item_structured_docs.append(structured_doc)
            start_idx += 1
        structured_doc_list.append(item_structured_docs)

    return structured_doc_list

def truncate_doc_content(self, doc_content: str, max_length: int = 25000) -> str:
    """
    Args:
        doc_content: str, the content of the document
        max_length: int, the max length of the document
    Output:
        str, the truncated document content
    """
    tokenized_content = tokenizer(doc_content, truncation=False, return_tensors="pt").input_ids[0]
    half = max_length // 2
    # truncate the middle content
    if len(tokenized_content) > max_length:
        doc_content = tokenizer.decode(
            tokenized_content[:half], skip_special_tokens=True
        ) + tokenizer.decode(tokenized_content[-half:], skip_special_tokens=True)
    return doc_content

### Adaptive Document Refinement

Based on the structured document tree, each node’s significance from both local and global perspectives are evaluated to adapt to varying information requirements and identify the most relevant information.


<b>Local Perspective.</b> From a local standpoint, relevant information for addressing the query may comprise discrete pieces distributed across multiple paragraphs, which correspond to the leaf nodes of the document tree. The refinement process is initiated by computing local score (LS) starting at the leaf nodes and subsequently propagating these scores upward through the tree hierarchy. The scoring mechanism is defined as follows:

$$
\text{LS}(n_i) = 
\begin{cases}
    \mathcal{M}(\text{query}, n_i) & \text{if } n_i \in \mathcal{N}_L \\
    \frac{1}{|C(n_i)|} \sum_{n_j \in C(n_i)}^{} \text{LS}(n_j) & \text{otherwise.}
\end{cases}
$$


Here, $\mathcal{M}$ represents a universal scoring model used to calculate the similarity between the query and each leaf node, providing the initial local score. $\mathcal{N}_L$ represents the set of all leaf nodes in the document tree. 

If the node is a paragraph at the bottom of the tree, its score is directly calculated by measuring how well it matches the user's query expressed as $\mathcal{M}(\text{query}, n_i)$.

If the node is a higher-level structural element (like a section or a subsection), its relevance score cannot be calculated directly from raw text. Instead, its score is the average Local Score of all its child nodes. $C(n_i)$ is the set of all children nodes belonging to the parent node $n_i$ (e.g., if $n_i$ is a "Section", its children might be several "Paragraphs"). Here, $∣C(n_i)∣$ is the total number of children belonging to node $n_i$ and $\sum_{n_j \in C(n_i)}^{} \text{LS}(n_j)$ is the sum of the Local Scores of all those child nodes. 

The local scores are propagated to parent nodes by averaging the scores of their child nodes. This method ensures that a parent node's score accurately reflects the overall quality of its content. Importantly, a parent node achieves a high local score only if all its child nodes maintain sufficiently high scores, thereby mitigating the risk of any single child node disproportionately affecting the parent's score.

<b>Global Perspective.</b> For queries that require a comprehensive understanding of the document, it is crucial to evaluate the importance of information from a global perspective. This prevents an overemphasis on localized information points, which could
lead to incomplete information retrieval. The computation of global scores is defined as follows:

$$
\text{GS}(n_i) = 
\begin{cases}
    \mathbb{I} (n_i \in \mathcal{M}(q, \text{outline})) & \text{if } n_i \in \mathcal{N}_S \\
    \frac{GS(Pa(n_i))}{|C(Pa(n_i))|} & \text{otherwise.}
\end{cases}
$$


Here, $\mathbb{I}(·)$ is the indicator function that outputs a 1 if the condition inside the parentheses is true, and a 0 if it is false. The $\mathcal{M}(q, \text{outline})$ denotes a model or matching function that takes the query $q$ and evaluates the entire document outline to select the most relevant structural sections. The $\mathcal{N}_S$ is a set of structural nodes (high-level nodes like chapters, sections, or subsections that appear in the document outline). Therefore, the first case means if the node is a high-level section/heading, its score is either 1 (if the model decides this section is relevant to the query outline) or 0 (if it's deemed irrelevant).

In the second case, $Pa$ represents the parent node, $\mathcal{N}_S$ represents the set of all section nodes in the document tree. The $Pa(n_i)$ is the Parent node of $n_i$ (the node immediately above it in the tree structure). The expression in the denominator $C(Pa(n_i))$ is the set of all Children belonging to that parent node, while $|C(Pa(n_i))|$ is the total <b>number</b> of sibling nodes (including $n_i$ itself) that share the same parent. This condition applies to text chunks or paragraphs that are not part of the high-level structural outline. The $\frac{GS(Pa(n_i))}{|C(Pa(n_i))|}$ takes the Global Score of the parent node and divides it equally among all of its children. If the node is a paragraph, it doesn't get a unique global score. Instead, it inherits a split share of its parent section's score. If a parent section has a score of 1 and contains 4 paragraphs, each paragraph receives a global score of $\frac{1}{4}=0.25$. If the parent section was deemed irrelevant (score 0), all its paragraphs inherit 0.

So, unlike the Local Score's bottom-up approach, the Global Score flows top-down. At the Top, the system looks at the big picture (the document outline) and assigns a 1 or 0 to major sections based on the query. Moving Down, a relevant section (1) filters its importance downward, dividing its score equally among its sub-sections or paragraphs. Its Purpose is that it penalizes paragraphs that might happen to contain a few keyword matches (high Local Score) but belong to a completely irrelevant section of the document overall, while prioritizing paragraphs that live inside highly relevant chapters.

The document outline consists of the abstract and the titles of all sections. By providing only the outline instead of the entire text, sufficient overall document information is supplied while preventing localized details from biasing the model's selection process. From a global perspective, each child node contributes equally to its parent node. Therefore, each child node is uniformly assigned to split its parent node's score equally to ensure every child node is considered from a global view.

In [ ]:
def run_global_selection(self, question_list: List[str], structured_doc_list: List[List[dict]]) -> List[List[dict]]:
    """
    Args:
        question_list: List[str], each question is a string
        structured_doc_list: List[List[dict]], each item is a list of documents, 
        each document is a dictionary, contains the structured content of the document
    Output:
        List[List[dict]], each item is a list of documents, each document is a dictionary, 
        contains the selected content of the document
    """
    # get prompt template, sampling params, lora request
    prompt_template = PromptTemplate(tokenizer, system_prompt=SYSTEM_PROMPT_STEP3, user_prompt=USER_PROMPT_STEP3)
    sampling_params = SamplingParams(temperature=0, max_tokens=10000)
    lora_request = LoRARequest(lora_name="doc_structuring", lora_int_id=2, lora_path=doc_structuring_module_lora_path)

    prompt_list = []
    for question, item_doc_list in zip(question_list, structured_doc_list):
        for doc in item_doc_list:
            abstract = doc.get("abstract", "")
            if abstract is None:
                abstract = ""
            elif isinstance(abstract, list):
                abstract = "\n".join(abstract)
            title = doc["title"]
            outline = f"Title: {title}\n"

            sections = doc["sections"]
            section_idx = 1
            for section_title, section_dict in sections.items():
                subsection_idx = 1
                if section_dict["content"] is not None or section_dict["subsections"] != {}:
                    outline += f"Section{section_idx}: {section_title}\n"
                    section_idx += 1
                if section_dict["subsections"] != {}:
                    for subsection, subsection_content in section_dict["subsections"].items():
                        outline += f"Subsection{subsection_idx}: {subsection}\n"
                        subsection_idx += 1

            prompt = prompt_template.get_prompt(question=question, abstract=abstract, outline=outline)
            prompt_list.append(prompt)

    output_list = model.generate(prompt_list, sampling_params=sampling_params, lora_request=lora_request)
    global_selection_result = []
    idx = 0
    for question, item_doc_list in zip(question_list, structured_doc_list):
        item_global_selection_result = []
        for doc in item_doc_list:
            selected_title = output_list[idx].outputs[0].text
            selected_title = json_repair.loads(selected_title)
            if isinstance(selected_title, dict):
                selected_title = selected_title.get("selected_titles", [])
            elif len(selected_title) > 1 and isinstance(selected_title[0], dict):
                selected_title = selected_title[0].get("selected_titles", [])
            selected_title = [i.lower() for i in selected_title]

            item_global_selection_result.append(selected_title)
            idx += 1
        global_selection_result.append(item_global_selection_result)
    return global_selection_result